# Giao diện tạo sinh được kiểm soát

Cho đến nay, AI agent của bạn chỉ có thể phản hồi bằng văn bản. Trong bài học này, chúng ta sẽ nâng cấp nó để hiển thị các giao diện React phong phú như: thẻ thông tin chuyến bay, biểu đồ tròn, hoặc bất kỳ thành phần nào bạn muốn tích hợp. 

Bạn sẽ đóng vai trò là người xây dựng các component, và agent sẽ tự động quyết định xem nên sử dụng component nào, vào lúc nào.

## 📋 Mục tiêu bài học
1. **Hiểu về GenUI:** Nắm bắt khái niệm GenUI và vị trí của "GenUI được kiểm soát".
2. **Đăng ký frontend component:** Sử dụng hook `useComponent()` để "phơi bày" các React component cho agent sử dụng.
3. **Hiển thị đầu ra có cấu trúc:** Cho phép agent lựa chọn và truyền dữ liệu vào các UI component ngay trong khung chat.

---

## 🎯 Sản phẩm thực hành

Bạn sẽ xây dựng một giao diện chat có khả năng hiển thị các UI component phong phú dựa trên yêu cầu của người dùng. 

**Ví dụ 1: Yêu cầu thông tin chuyến bay**
> 🗣️ **Người dùng:** *Hiển thị thông tin chuyến bay của hãng Pacific Air từ SFO đến JFK cất cánh lúc 08:30 với giá $249*
> 
> 🤖 **Agent:** (Hiển thị một thẻ UI đẹp mắt chứa mã chuyến bay, giờ khởi hành, điểm đi/đến và giá vé).

<img src="images/flight-card.png" style="display: block; margin: 0 auto; max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

**Ví dụ 2: Yêu cầu xem báo cáo dữ liệu**
> 🗣️ **Người dùng:** *Hiển thị cơ cấu doanh thu theo danh mục bằng biểu đồ tròn*
> 
> 🤖 **Agent:** (Gọi công cụ phân tích dữ liệu từ file CSV, sau đó hiển thị một biểu đồ tròn trực quan minh họa doanh thu theo từng danh mục).

<img src="images/pie-chart.png" style="display: block; margin: 0 auto; max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

---

## 📖 GenUI được kiểm soát là gì?

**GenUI** là một mẫu thiết kế nơi các agent phản hồi bằng các giao diện tương tác hoàn chỉnh thay vì chỉ là văn bản thuần túy. 

**GenUI được kiểm soát** là biến thể được giới hạn chặt chẽ nhất của phương pháp này: Agent **chỉ có thể** render các component mà bạn đã đăng ký một cách rõ ràng.

### Cách thức hoạt động
Mỗi component được đăng ký sẽ đóng vai trò như một **công cụ** đối với agent. Mỗi công cụ này bao gồm:
- Một cái tên cố định.
- Một schema đầu vào.
- Một React component tương ứng được ánh xạ.

Agent sẽ không tự do tạo ra một UI bất kỳ. Thay vào đó, nó sẽ truyền dữ liệu có cấu trúc vào các component mà bạn đã xây dựng sẵn trên frontend và đã đăng ký lúc chạy (runtime).

### Ưu và nhược điểm

**✅ Ưu điểm:**
- **Dễ triển khai:** Chỉ cần đăng ký component và bạn đã hoàn tất.
- **Độ hoàn thiện cao:** Giao diện hiển thị chính là giao diện bạn đã cẩn thận lập trình.
- **Tính an toàn cao:** Mô hình AI chỉ có thể gọi các công cụ đã đăng ký với các tham số đã được xác thực.
- **Phù hợp cho môi trường thực tế:** Rất tốt cho các dự án có lưu lượng truy cập lớn hoặc UX đòi hỏi tính ổn định cao.

**❌ Nhược điểm:**
- Khối lượng công việc frontend sẽ tăng lên theo mỗi tính năng mới: Mỗi một loại dữ liệu mới đều cần một component riêng.
- Hạn chế sự tự do sáng tạo so với các phương pháp GenUI mang tính khai báo hoặc mở.

---

## 💻 Hook `useComponent()`

Trong hệ sinh thái CopilotKit, hook `useComponent` được dùng để đăng ký một React component như một công cụ mà agent có thể gọi bên trong `<CopilotChat />`. Bạn định nghĩa những gì có sẵn, agent sẽ chọn thời điểm sử dụng.

**Cú pháp cơ bản:**
```tsx
useComponent({
  name: "component_name",
  description: "Mô tả để agent biết khi nào nên dùng component này",
  parameters: z.object({ ... }),
  render: MyComponent,
});
```

- `name` *(string, bắt buộc)*: Tên công cụ phơi bày cho mô hình AI.
- `description` *(string, tùy chọn)*: Chỉ dẫn cho mô hình biết khi nào nên gọi công cụ này.
- `parameters` *(Zod schema, tùy chọn)*: Định dạng cấu trúc dữ liệu sẽ được truyền vào làm đối số.
- `render` *(bắt buộc)*: Một React component (sẽ được render dạng `<Component {...args} />`), hoặc một hàm nhận `{ args, status }` nếu bạn muốn custom hiển thị (như trạng thái loading).

---

## ⚙️ Hướng dẫn cài đặt & khởi chạy

### Khởi tạo môi trường & backend
Backend của chúng ta (được định nghĩa trong `server.py`) sử dụng `LangGraph` và `FastAPI`. Agent được lập trình sẵn công cụ `query_data` (đọc file `db.csv`) và được hướng dẫn sử dụng công cụ `pieChart` cho phân phối danh mục, và `flightCard` cho tóm tắt chuyến bay.

Để bắt đầu, hãy tải các biến môi trường và chạy backend:

In [1]:
# Tải các API key từ file .env
from helper import load_api_keys
load_api_keys()

# Khởi chạy backend agent ở port 8003
from backend.server import start_backend
start_backend(port=8003)

✓ OpenAI API key loaded
✓ Google API key loaded
✓ Server running at http://localhost:8003


### Khởi chạy frontend
Frontend sẽ hiển thị giao diện và kết nối với backend thông qua CopilotKit.

In [2]:
from helper import start_frontend
start_frontend(port=3003)

Starting frontend on port 3003 ...
✓ App running at http://localhost:3003

Read the logs: /home/cuong-ta/Documents/build-interactive-agents-with-generative-ui/2-controlled-generative-ui/frontend/dev-logs.txt


---

## 🚀 Đăng ký component (Thực hành frontend)

Chúng ta sử dụng `useComponent()` để đăng ký 3 component: 
1. `showMyName`: Một thẻ đơn giản hiển thị tên.
2. `pieChart`: Biểu đồ tròn hiển thị dữ liệu kinh doanh.
3. `flightCard`: Thẻ hiển thị thông tin chuyến bay.

In [3]:
%%writefile frontend/src/App.tsx
import { z } from "zod"
import { CopilotChat } from "@copilotkit/react-core/v2";
import { useComponent } from "@copilotkit/react-core/v2";

import { FlightCard, FlightCardProps } from "@/components/flight-card";
import { PieChart, PieChartProps } from "@/components/pie-chart";

import { useExampleSuggestions } from "@/hooks/use-example-suggestions";

export default function App() {

  // 🪁 Đăng ký component hiển thị tên người dùng
  useComponent({
    name: "showMyName",
    description: "Hiển thị tên của người dùng trong một thẻ.",
    parameters: z.object({ name: z.string() }),
    render: ({ name }) => <div className="bg-blue-500 p-4">Hi, {name}!</div>,
  });

  // 🪁 Đăng ký component pieChart để hiển thị dữ liệu cấu trúc
  useComponent({
    name: "pieChart",
    description: "Hiển thị dữ liệu dưới dạng biểu đồ hình tròn.",
    parameters: PieChartProps,
    render: PieChart,
  });

  // 🪁 Đăng ký component flightCard để hiển thị dữ liệu chuyến bay
  useComponent({
    name: "flightCard",
    description: "Hiển thị thẻ tóm tắt thông tin một chuyến bay.",
    parameters: FlightCardProps,
    render: FlightCard,
  });

  // 🪁 Thêm các gợi ý prompt cho người dùng hiển thị dưới dạng nút bấm
  useExampleSuggestions();

  return <CopilotChat />;

};

Overwriting frontend/src/App.tsx


Sau khi chạy ứng dụng, bạn có thể kiểm thử agent bằng cách nhập các câu lệnh sau vào khung chat:
- *"Hiển thị tên tôi"* (Agent sẽ hỏi tên bạn và trả về giao diện khung màu xanh).
- *"Biểu đồ hình tròn"* (Agent sẽ đọc dữ liệu từ `db.csv` và vẽ biểu đồ).
- *"Thẻ chuyến bay"* (Agent sẽ điền dữ liệu giả lập hoặc yêu cầu bạn cung cấp thông tin chuyến bay để tạo thẻ).

---

## 🔍 Nhìn lại: Cấu trúc của một công cụ UI & Thư viện Zod

Bất kỳ React component nào cũng có thể trở thành một công cụ GenUI. Hãy xem ví dụ đầy đủ của component `FlightCard`:

```tsx
import { z } from "zod";

// 1. Định nghĩa Zod schema (vừa là xác thực đầu vào, vừa là hướng dẫn cho LLM)
export const FlightCardProps = z.object({
  title: z.string().describe("Tiêu đề thẻ chuyến bay"),
  airline: z.string().describe("Tên hãng hàng không"),
  origin: z.string().describe("Sân bay hoặc thành phố xuất phát"),
  destination: z.string().describe("Sân bay hoặc thành phố điểm đến"),
  departure_time: z.string().describe("Thời gian khởi hành"),
  price: z.string().describe("Giá vé hiển thị"),
});

// 2. Xuất kiểu dữ liệu TypeScript từ Zod schema
type FlightCardProps = z.infer<typeof FlightCardProps>;

// 3. Xây dựng UI component
export function FlightCard({
  title, airline, origin, destination, departure_time, price,
}: FlightCardProps) {
  return (
    <div className="rounded-lg border bg-white p-3 space-y-2">
      <div className="font-semibold">{title}</div>
      <div className="rounded border p-2 text-sm">
        <div className="font-medium">{airline}</div>
        <div>
          {origin} → {destination}
        </div>
        <div>Departs: {departure_time}</div>
        <div className="font-semibold text-xl mt-2">{price}</div>
      </div>
    </div>
  );
}
```

**💡 Tại sao lại dùng Zod?**
[Zod](https://zod.dev) là một thư viện xây dựng schema và xác thực dữ liệu tối ưu cho TypeScript (tương tự như `Pydantic` trong Python). Nó cho phép bạn định nghĩa các props của component một lần duy nhất và tái sử dụng schema đó cho cả:
1. Định nghĩa Type trong TypeScript.
2. Cung cấp tham số (`parameters`) cho `useComponent()`, giúp LLM biết chính xác dữ liệu nào cần được sinh ra (`describe`).

---

## 🎓 Tổng kết bài học

- **GenUI được kiểm soát** giới hạn việc sinh giao diện bằng các component đã được đăng ký cụ thể, giúp đảm bảo giao diện luôn chuẩn xác và an toàn.
- Hook `useComponent()` tạo ra một "bản hợp đồng" về dữ liệu giữa agent và React component.
- **Sự phân tách rõ ràng:** Backend và frontend chia sẻ trách nhiệm rất gọn gàng. Agent (backend) chọn công cụ và chuẩn bị dữ liệu, trong khi UI layer (frontend) làm chủ việc hiển thị.
- Phương pháp này tạo ra trải nghiệm an toàn, có thể dự đoán được cho người dùng, dù phải hy sinh một chút sự linh hoạt của AI.

**🛠️ Bài tập thử thách:** 
Bạn hãy thử tạo một component mới (ví dụ: Một bảng dữ liệu `Table` hoặc thanh tiến độ `ProgressBar`), định nghĩa Zod schema cho nó, đăng ký bằng `useComponent` và yêu cầu agent sử dụng nó!

➡️ **Bước tiếp theo:** Trong bài tiếp theo, bạn sẽ tiến tới **GenUI mang tính khai báo** - nằm ở giữa phổ linh hoạt của GenUI. Thay vì đăng ký từng component riêng lẻ, bạn sẽ định nghĩa một thư viện các "viên gạch xây dựng" và để agent tự do lắp ghép chúng thành các bố cục phức tạp!